In [ ]:
!pip install wfdb neurokit2 biosppy pywavelets antropy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.5/159.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 93.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [ ]:
import sys

PROJECT_PATH = "/content/drive/MyDrive/DL & ML/features"

sys.path.append(PROJECT_PATH)

import os
import pandas as pd
from tqdm import tqdm

# Your modules
from load_record import ECGRecord
from preprocessing import preprocess_signal
from feature_extractor import extract_morphologymorphology
from feature_extractor import extract_hrvhrv
from feature_extractor import extracstatistical
from feature_extractor import frequency
from feature_extractor import nonlinear

ImportError: cannot import name 'morphology' from 'feature_extractor' (/content/drive/MyDrive/DL & ML/features/feature_extractor.py)

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/ML_MODEL/ECG_DATA/CHD"

OUTPUT_CSV = "/content/drive/MyDrive/ML_MODEL/features"

In [ ]:
def extract_features(record_path, label):

    # Load ECG
    record = ECGRecord(record_path)

    signal = record.signal
    fs = record.fs

    # Preprocess
    filtered_signal = preprocess_signal(signal, fs)

    # Detect R peaks
    r_peaks = detect_r_peaks(filtered_signal, fs)

    # RR intervals
    rr_intervals = (r_peaks[1:] - r_peaks[:-1]) / fs

    features = {}

    # Record information
    features["record"] = os.path.basename(record_path)
    features["label"] = label

    # Morphology
    features.update(
        morphology.extract(filtered_signal, fs, r_peaks)
    )

    # HRV
    features.update(
        hrv.extract(rr_intervals)
    )

    # Statistics
    features.update(
        statistical.extract(filtered_signal)
    )

    # Frequency
    features.update(
        frequency.extract(filtered_signal, fs)
    )

    # Nonlinear
    features.update(
        nonlinear.extract(rr_intervals)
    )

    return features

In [ ]:
dataset = []

for label_name in ["NORMAL", "CHD"]:

    folder = os.path.join(DATASET_PATH, label_name)

    label = 0 if label_name == "NORMAL" else 1

    for file in os.listdir(folder):

        if file.endswith(".hea"):

            record_path = os.path.join(folder, file[:-4])

            dataset.append((record_path, label))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DL & ML/ECG_DATA/CHD/NORMAL'

In [ ]:
all_features = []

for record_path, label in tqdm(dataset):

    try:

        features = extract_features(record_path, label)

        all_features.append(features)

    except Exception as e:

        print(record_path)
        print(e)

In [ ]:
df = pd.DataFrame(all_features)

print(df.shape)

df.head()

In [ ]:
df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("CSV saved.")

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/ML_MODEL/features"))